In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

# =========================================================
# Session 1: Load & Data Cleaning
# =========================================================
def load_and_prep(filename):
    df = pd.read_csv(filename)
    df.columns = df.columns.str.strip()
    # ล้างช่องว่างในข้อมูล String ทั้งหมด
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip()
    return df

inv_df = load_and_prep('01-inventory.csv')
recipe_df = load_and_prep('02-recipe.csv')
recipe_sum_df = load_and_prep('05-recipe-summary.csv')
orders_df = load_and_prep('06-orders-time-bucket.csv')
git_df = load_and_prep('07-git-time-buckets.csv')

# แปลงตัวเลขให้ถูกต้อง (จัดการเครื่องหมาย comma)
inv_df['stock_qty'] = pd.to_numeric(inv_df['stock_qty'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
orders_df['order_qty'] = pd.to_numeric(orders_df['order_qty'], errors='coerce').fillna(0)
orders_df['bucket'] = pd.to_numeric(orders_df['bucket'], errors='coerce').fillna(0).astype(int)
git_df['git_qty'] = pd.to_numeric(git_df['git_qty'], errors='coerce').fillna(0)
git_df['bucket'] = pd.to_numeric(git_df['bucket'], errors='coerce').fillna(0).astype(int)
recipe_df['usage_percent'] = pd.to_numeric(recipe_df['usage_percent'], errors='coerce').fillna(0)
recipe_sum_df['total_cost'] = pd.to_numeric(recipe_sum_df['total_cost'], errors='coerce').fillna(0)

# =========================================================
# Session 2: Dictionaries & Variable Mapping
# =========================================================
rm_initial_inv = inv_df.set_index('rm_name')['stock_qty'].to_dict()
all_rms = sorted(list(rm_initial_inv.keys()))
git_grouped = git_df.groupby(['rm_name', 'bucket'])['git_qty'].sum().to_dict()

product_to_recipes = recipe_sum_df.groupby('product_name')['recipe_number'].apply(list).to_dict()
recipe_usage = recipe_df.set_index(['recipe_number', 'rm_name'])['usage_percent'].to_dict()
recipe_costs = recipe_sum_df.set_index('recipe_number')['total_cost'].to_dict()
recipe_total_usage_pct = recipe_sum_df.set_index('recipe_number')['total_usage_pct'].to_dict()

all_buckets = sorted(list(set(orders_df['bucket'].unique()) | set(git_df['bucket'].unique())))

# สร้างรายการตัวตัดสินใจ (Decision Variables)
mo_choices = []
for idx, row in orders_df.iterrows():
    recs = product_to_recipes.get(row['product_name'], [])
    for r in recs:
        mo_choices.append((idx, r))
    mo_choices.append((idx, 'NONE')) # ทางเลือก "ไม่ผลิต"

n_vars = len(mo_choices)

# =========================================================
# Session 3: ปรับปรุงความเร็ว (Optimized Constraints)
# =========================================================
print("Building constraints... this should be faster now.")

# 1. คัดเลือกเฉพาะ RM ที่มีการใช้งานจริงใน recipe_usage
active_rms = {rm for (rec, rm) in recipe_usage.keys()}
# กรองอีกชั้น เอาเฉพาะที่มีใน inventory จริงๆ
active_rms = [rm for rm in all_rms if rm in active_rms]

A_rows = []
lb = []
ub = []

# C1: 1 Choice per MO (ทำเหมือนเดิมแต่เขียนให้กระชับ)
for m_idx in orders_df.index:
    row = np.zeros(n_vars)
    indices = [i for i, (v_idx, r_name) in enumerate(mo_choices) if v_idx == m_idx]
    row[indices] = 1
    A_rows.append(row); lb.append(1); ub.append(1)

# C2: Cumulative Inventory (เช็คเฉพาะ RM ที่ใช้งานจริง)
# และสร้าง Matrix ล่วงหน้าเพื่อลดภาระการวนลูป
for rm_name in active_rms:
    # คำนวณ Supply สะสมรอไว้ก่อน (Vectorized calculation)
    supply_by_bucket = {}
    cum_git = rm_initial_inv.get(rm_name, 0)

    for b_val in all_buckets:
        cum_git += git_grouped.get((rm_name, b_val), 0)

        row = np.zeros(n_vars)
        # เติมค่าเฉพาะตัวแปรที่มีการใช้ RM นี้
        relevant_vars_found = False
        for i, (m_idx, r_name) in enumerate(mo_choices):
            if r_name != 'NONE':
                usage = recipe_usage.get((r_name, rm_name), 0)
                if usage > 0 and orders_df.loc[m_idx, 'bucket'] <= b_val:
                    row[i] = orders_df.loc[m_idx, 'order_qty'] * usage
                    relevant_vars_found = True

        if relevant_vars_found:
            A_rows.append(row)
            lb.append(-np.inf)
            ub.append(cum_git)

# แปลงเป็น Matrix ครั้งเดียว
A_final = np.array(A_rows)
print(f"Matrix built: {A_final.shape}")


print("--- Matrix Dimension Check ---")
print(f"Total Decision Variables (Columns): {n_vars}")
print(f"Total Constraints (Rows): {A_final.shape[0]}")
print(f"Total Elements in Matrix: {A_final.size:,}")

# เช็กความหนาแน่น (Density) - Matrix ที่ดีควรจะมีค่าส่วนใหญ่เป็น 0 (Sparse)
non_zero = np.count_nonzero(A_final)
density = (non_zero / A_final.size) * 100
print(f"Matrix Density: {density:.4f}%")

# Run MILP
res = milp(c=c, constraints=LinearConstraint(np.array(A_rows), lb, ub),
           integrality=np.ones(n_vars), bounds=Bounds(0, 1))

# =========================================================
# Session 4: Generate Reports (07, 08, 09, 11)
# =========================================================
selected = {m_idx: r_name for i, (m_idx, r_name) in enumerate(mo_choices) if res.x[i] > 0.5}

# --- 07-orders-time-bucket-calculation.csv ---
out_07 = orders_df.copy()
out_07['recipe_number'] = out_07.index.map(lambda i: selected[i] if selected[i] != 'NONE' else None)
out_07['is_producible'] = out_07.index.map(lambda i: True if selected[i] != 'NONE' else False)
out_07['recipe_cost'] = out_07['recipe_number'].map(recipe_costs).fillna(0)
out_07['recipe_usage_pct'] = out_07['recipe_number'].map(recipe_total_usage_pct).fillna(0)
out_07['order_cost'] = out_07['order_qty'] * out_07['recipe_cost']
out_07['remark'] = out_07['is_producible'].map({True: "OK", False: "RM shortage (skipped)"})
out_07.to_csv('07-orders-time-bucket-calculation.csv', index=False)

# --- 08-inventory-calculation.csv & 09-inventory-summary.csv ---
res_08 = []; rm_usage_totals = {rm: 0 for rm in all_rms}
for rm in all_rms:
    cur = rm_initial_inv.get(rm, 0)
    for b in all_buckets:
        git = git_grouped.get((rm, b), 0)
        use = sum(orders_df.loc[m, 'order_qty'] * recipe_usage.get((r, rm), 0)
                  for m, r in selected.items() if r != 'NONE' and orders_df.loc[m, 'bucket'] == b)
        rm_usage_totals[rm] += use
        after = cur + git - use
        res_08.append({'rm_name': rm, 'bucket': b, 'opening_qty': cur, 'git_qty': git,
                       'usage_by_select_recipe': use, 'total_qty_after_git_usage': after,
                       'utilization_pct': (use / (cur+git))*100 if (cur+git)>0 else 0})
        cur = after

pd.DataFrame(res_08).to_csv('08-inventory-calculation.csv', index=False)

# --- 09-inventory-summary.csv (สรุปราย RM) ---
pd.DataFrame([{
    'rm_name': rm,
    'total_available': rm_initial_inv[rm] + sum(git_grouped.get((rm, b), 0) for b in all_buckets),
    'total_usage': rm_usage_totals[rm],
    'utilization_pct': (rm_usage_totals[rm] / (rm_initial_inv[rm] + sum(git_grouped.get((rm, b), 0) for b in all_buckets))) * 100
    if (rm_initial_inv[rm] + sum(git_grouped.get((rm, b), 0) for b in all_buckets)) > 0 else 0
} for rm in all_rms]).to_csv('10-inventory-summary.csv', index=False)

# --- 11-mo-shortage-details.csv (รายละเอียดที่ขาดแยกตาม RM) ---
shortage_details = []
for idx, row in out_07[out_07['is_producible'] == False].iterrows():
    mo_info = row.to_dict()
    recs = product_to_recipes.get(row['product_name'], [])
    r_best = sorted(recs, key=lambda r: recipe_costs.get(r, 0))[0] if recs else "N/A"
    mo_info['target_recipe'] = r_best
    if r_best != "N/A":
        for rm in all_rms:
            req = row['order_qty'] * recipe_usage.get((r_best, rm), 0)
            if req > 0:
                supply = rm_initial_inv[rm] + sum(git_grouped.get((rm, bb), 0) for bb in all_buckets if bb <= row['bucket'])
                used = sum(orders_df.loc[m, 'order_qty'] * recipe_usage.get((selected[m], rm), 0)
                           for m in selected if selected[m] != 'NONE' and orders_df.loc[m, 'bucket'] <= row['bucket'])
                short_val = max(0, req - (supply - used))
                if short_val > 0: mo_info[f'short_amount_{rm}'] = short_val
    shortage_details.append(mo_info)
pd.DataFrame(shortage_details).to_csv('11-mo-shortage-details.csv', index=False)

# --- 11-summary.csv (บทสรุปภาพรวม) ---
total_produced_qty = out_07[out_07['is_producible'] == True]['order_qty'].sum()
total_produced_cost = out_07[out_07['is_producible'] == True]['order_cost'].sum()
total_orders = len(out_07)
producible_count = len(out_07[out_07['is_producible'] == True])

sum_metrics = [
    ["Total Produced Quantity", total_produced_qty],
    ["Total Order Cost (Produced)", total_produced_cost],
    ["Total Orders Count", total_orders],
    ["Producible Orders (Count)", producible_count],
    # แก้ไขจุดนี้: ใช้ต้นทุนที่ผลิตได้จริง หารด้วย จำนวนที่ผลิตได้จริง
    ["Production Cost (Baht per Ton)", total_produced_cost / total_produced_qty if total_produced_qty > 0 else 0],
    ["Non-Producible Orders (Count)", total_orders - producible_count],
    ["Success Rate (%)", (producible_count / total_orders) * 100 if total_orders > 0 else 0]
]
pd.DataFrame(sum_metrics, columns=["Metric", "Value"]).to_csv('09-summary.csv', index=False)

print("Status:", res.message)
print("Reports generated: 08-orders-time-bucket-calculate, 09-summary, 10-inventory-summary , 11-mo-shortage-details")